# Silver · custos

**Origem:** `vcredit_bronze.custos` → **Destino:** `vcredit_silver.custos`

**Grão:** uma linha por registro de custo. **Depende de:** bronze

Custo, em reais, de cada atendimento.

| Regra | Tratamento |
|---|---|
| `id_custo` e `id_chamado` | Convertidos para número |
| `id_custo` repetido | Mantém um registro |
| `custo` | Remove "R$", "reais" e outros textos; vírgula vira ponto (separador de centavos); convertido para decimal com 2 casas. Valor que não vira número fica nulo |

**Decisões:** o valor de custo vinha em formatos diferentes ("x,xx", "R$ x,xx", "x,xxreais"); todos são tratados como reais com centavos. Na gold, cada chamado fica com um único custo.

Pipeline idempotente (lê da bronze e sobrescreve).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "rocket_lab"

bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

silver_schema_name = "vcredit_silver"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

In [0]:
# Importação da tabela custos da camada bronze
bronze_custos = spark.table(f"{bronze_schema}.custos")

# Tratamento e padronização dos dados da tabela custos
silver_custos = (
    bronze_custos
    .withColumn("id_custo", F.expr("try_cast(trim(id_custo) as bigint)"))
    .withColumn("id_chamado", F.expr("try_cast(trim(id_chamado) as bigint)"))
    .dropDuplicates(["id_custo"])
    .withColumn("custo", F.regexp_replace(F.trim("custo"), r"[^0-9,.\-]", "")) 
    .withColumn("custo", F.regexp_replace("custo", ",", "."))                  
    .withColumn("custo", F.expr("try_cast(custo as decimal(18,2))"))
)

 #idempotente
(
    silver_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.custos")
)

display(silver_custos.orderBy("id_custo").limit(10))